In [ ]:
# %% [markdown]
# # TF-IDF feature extraction only (no training)
# Fits on the TRAIN split of dataset_main.csv, then transforms
# main (train/val/test) and the three stress sets with the same vocabulary.

# %%
import os
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR = "../Data"
OUT_DIR = "../Data/TFIDFV11"
MODEL_DIR = "models"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN = "train"   # value in the `split` column used for fitting

# Also write human-readable dense CSVs like your features_*.csv files?
# Only sensible with a SMALL vocabulary (see MAX_* below), otherwise the
# files become enormous. The .npz sparse files are always saved.
SAVE_CSV = False
MAX_WORD = 5_000 if SAVE_CSV else 200_000
MAX_CHAR = 5_000 if SAVE_CSV else 300_000

STRESS = ["test_notinject", "test_obfuscated", "test_temporal"]

# %% [markdown]
# ## Load

# %%
def load(name):
    df = pd.read_csv(f"{DATA_DIR}/{name}.csv")
    df["text"] = df["text"].fillna("").astype(str)
    return df

main = load("dataset_main")
sets = {"main": main}
for s in STRESS:
    sets[s] = load(s)

train_text = main.loc[main["split"] == TRAIN, "text"]
print("train rows:", len(train_text))
print(main["split"].value_counts())

# %% [markdown]
# ## Fit vectorizers on train only

# %%
word_vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), lowercase=True,
    min_df=2, max_df=0.95, sublinear_tf=True, max_features=MAX_WORD,
    token_pattern=r"(?u)\b\w+\b", dtype=np.float32,
)
char_vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), lowercase=True,
    min_df=3, sublinear_tf=True, max_features=MAX_CHAR,
    dtype=np.float32,
)

word_vec.fit(train_text)
char_vec.fit(train_text)

feature_names = (
    ["w_" + f for f in word_vec.get_feature_names_out()]
    + ["c_" + f for f in char_vec.get_feature_names_out()]
)
print("total features:", len(feature_names))

joblib.dump(word_vec, f"{OUT_DIR}/tfidf_wordV1.joblib")
joblib.dump(char_vec, f"{OUT_DIR}/tfidf_charV1.joblib")
pd.Series(feature_names, name="feature").to_csv(
    f"{OUT_DIR}/feature_names.csv", index=False)

# %% [markdown]
# ## Transform + save every dataset

# %%
meta_cols = ["uid", "split", "label", "attack_type", "family_id", "source"]

for name, df in sets.items():
    X = sp.hstack([word_vec.transform(df["text"]),
                   char_vec.transform(df["text"])]).tocsr()

    # Sparse matrix (compact; use this for training later)
    sp.save_npz(f"{OUT_DIR}/X_{name}.npz", X)

    # Row-aligned metadata so you can map rows back to samples
    keep = [c for c in meta_cols if c in df.columns]
    df[keep].to_csv(f"{OUT_DIR}/meta_{name}.csv", index=False)

    # Optional dense CSV (small vocab only)
    if SAVE_CSV:
        out = pd.DataFrame(X.toarray(), columns=feature_names)
        out.insert(0, "label", df["label"].values)
        out.insert(0, "uid", df["uid"].values)
        out.to_csv(f"{OUT_DIR}/features_tfidf_{name}.csv", index=False)

    print(f"{name:18s} X shape={X.shape}  nnz={X.nnz:,}")

# %% [markdown]
# ## Later: reload for training
# ```python
# X_all  = sp.load_npz("Data/tfidf/X_main.npz")
# meta   = pd.read_csv("Data/tfidf/meta_main.csv")
# X_train = X_all[(meta["split"] == "train").values]
# y_train = meta.loc[meta["split"] == "train", "label"].values
# ```

train rows: 9634
split
train    9634
test     1928
val      1927
Name: count, dtype: int64
total features: 211448
main               X shape=(13489, 211448)  nnz=9,951,631
test_notinject     X shape=(339, 211448)  nnz=60,429
test_obfuscated    X shape=(1800, 211448)  nnz=111,646
test_temporal      X shape=(485, 211448)  nnz=1,596,828
